# SRV02 and ROTFLEX Control Systems

<p align="center">
  <img src="../assets/diagrams/project-overview.svg" width="920" alt="SRV02 and ROTFLEX control-system overview">
</p>

This notebook is the main technical walkthrough for two related MATLAB and
Simulink control studies:

1. Position control of the Quanser SRV02 rotary servo using LQR,
   Ziegler-Nichols reaction-curve tuning, and relay auto-tuning.
2. Direct digital control of the SRV02-ROTFLEX flexible joint using
   state feedback, integral action, and numerical state estimation.

The implementation language is MATLAB. The notebook uses MATLAB code cells
and does not replace the original implementation with Python.

## Evidence and provenance convention

The project contains several kinds of technical evidence. They are kept
distinct throughout this notebook:

- **Verified implementation:** equations, constants, algorithms, and block
  settings that can be read directly from the MATLAB or Simulink files.
- **Recorded result:** a plot or numerical metric captured during the
  original experiment or model run. Raw logs are not available, so these
  results are preserved but are not claimed to be recomputed here.
- **Reconstructed calculation:** a deterministic value that follows from
  the stored model parameters, such as the discrete matrices or pole-
  placement gain.
- **Supplementary engineering explanation:** context added to connect the
  stored theory and code. It is not presented as an additional experiment.
- **Supplemental simulation:** an optional offline analysis with explicitly
  stated assumptions or synthetic noise.

## 1. Hardware and signal flow

The flexible-joint workflow targets the following components:

1. **SRV02 rotary servo:** the actuated motor and load shaft.
2. **ROTFLEX or ROTFLEX-E:** the compliant arm and spring assembly.
3. **Q2-USB:** the QUARC hardware interface used by the stored model.
4. **Position sensing:** potentiometer or quadrature encoder, selected in
   the model and converted to radians by calibration gains.
5. **Amplifier:** VoltPAQ or UPM configuration, with the model limiting the
   data-acquisition command to approximately $\pm10$ V.
6. **Measured output:** $y=[\theta,\alpha]^T$, containing the motor/load
   angle and the flexible-joint deflection.

<p align="center">
  <img src="../assets/diagrams/flexible-joint-plant-model.jpg" width="920" alt="SRV02 and ROTFLEX plant-interface model">
</p>

The red unresolved-library markers in this archived screenshot occur when
the Quanser QUARC library is unavailable. They are dependency indicators,
not missing project logic. The stored HIL analog-output block is disabled;
hardware output must only be enabled after the wiring, sensor direction,
amplifier selection, voltage limits, and emergency stop have been checked.

# Part I - SRV02 position control

## 2. Identified plant model

### Theory

The measured load-speed model is approximated by

$$
\frac{\Omega_l(s)}{V_m(s)}=\frac{K}{\tau s+1},
$$

where $V_m$ is the motor voltage, $\Omega_l$ is load angular velocity,
$K$ is the steady-state speed gain, and $\tau$ is the time constant.

Position adds an integrator:

$$
\frac{\Theta_l(s)}{V_m(s)}=\frac{K}{s(\tau s+1)}.
$$

The stored relay model uses $K=1.5286$ and $\tau=0.0254$ s.

In [ ]:
%% MATLAB implementation - identified SRV02 plant
tau = 0.0254;       % s
K = 1.5286;         % (rad/s)/V
if isfolder(fullfile(pwd, 'SRV02-Position-Control'))
    repositoryRoot = pwd;
else
    repositoryRoot = fileparts(pwd);
end
positionControlDir = fullfile(repositoryRoot, 'SRV02-Position-Control');
addpath(positionControlDir);
s = tf('s');

speedPlant = K/(tau*s + 1);
positionPlant = K/(s*(tau*s + 1));

### Computed result and interpretation

Executing the preceding MATLAB cell constructs the identified speed and
position transfer functions with $K=1.5286$ and $\tau=0.0254$ s. The
position plant contains one additional pole at the origin, which explains why
position tracking requires feedback and integral action. These values are
model parameters stored with the project, not newly measured data.

## 3. Linear Quadratic Regulator

### Theory

LQR selects the feedback gain that minimizes

$$
J=\int_0^\infty\left(x^TQx+u^TRu\right)dt.
$$

The implemented design uses a three-state position/velocity/integral
representation:

$$
A=\begin{bmatrix}
0&1&0\\
0&-1/\tau&0\\
1&0&0
\end{bmatrix},\qquad
B=\begin{bmatrix}0\\K/\tau\\0\end{bmatrix},
$$

with $Q=\operatorname{diag}(30,0,3)$ and $R=1$. A larger state weight
penalizes the associated state more strongly; a larger $R$ penalizes
motor effort. The continuous-time algebraic Riccati equation produces
$P$, and the optimal feedback law is

$$
u(t)=-K_{LQR}x(t),\qquad K_{LQR}=R^{-1}B^TP.
$$

The initial documented calculation produced approximately
$[K_p,K_v,K_i]=[11.79,0.23,1.73]$. The gain used in the recorded tests
was adjusted separately and is reported with the results below. The input
matrix here uses $K/\tau$, which is the realization consistent with the
identified transfer function $K/(\tau s+1)$.

In [ ]:
%% MATLAB implementation - LQR gain and poles
A_lqr = [0, 1,      0;
         0, -1/tau, 0;
         1, 0,      0];
B_lqr = [0; K/tau; 0];
Q_lqr = diag([30, 0, 3]);
R_lqr = 1;

[K_lqr, S_lqr, poles_lqr] = lqr(A_lqr, B_lqr, Q_lqr, R_lqr);
K_lqr
poles_lqr

### Computed and recorded results

The preceding cell computes the model-based gain and closed-loop poles when MATLAB is available. The controller used for the recorded tests was tuned to approximately
$K_p=10$, $K_v=0.31$, and $K_i=3.88$. The recorded step response has a
rise time of 0.174 s and a reported 5% settling time of about 0.758 s.
The control voltage remains below the 10 V operating limit.

| Step response | Step control voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/09-lqr-step-response.jpg" width="430"> | <img src="../assets/results/position-control/10-lqr-step-control-voltage.png" width="430"> |

The ramp test has a recorded steady tracking difference of approximately
0.064 rad. The disturbance test shows recovery to the command, but more
slowly than the later tuned controllers.

| Ramp response | Ramp control voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/11-lqr-ramp-response.jpg" width="430"> | <img src="../assets/results/position-control/12-lqr-ramp-control-voltage.png" width="430"> |

| Disturbance response | Disturbance control voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/13-lqr-disturbance-response.png" width="430"> | <img src="../assets/results/position-control/14-lqr-disturbance-control-voltage.png" width="430"> |

**Interpretation.** LQR produced the most conservative voltage demand and
good stability without repeated empirical retuning. Its recorded transient
was slower than the final Ziegler-Nichols and relay-tuned controllers.

## 4. Ziegler-Nichols reaction-curve tuning

### Theory

The maximum-slope tangent is used to estimate two reaction-curve
parameters. With tangent slope $R_m$, tangent point $(t_0,y_0)$, the
stored convention is

$$
a=|R_mt_0-y_0|,\qquad
\tau_{ZN}=\left|t_0-\frac{y_0}{R_m}\right|.
$$

The PID table used in the project is

| Controller | $K_p$ | $T_i$ | $T_d$ |
|---|---:|---:|---:|
| P | $1/a$ | - | - |
| PI | $0.9/a$ | $3\tau_{ZN}$ | - |
| PID | $1.2/a$ | $2\tau_{ZN}$ | $0.5\tau_{ZN}$ |

<p align="center">
  <img src="../assets/results/position-control/08-zn-tangent-method.png" width="520" alt="Reaction-curve tangent construction">
</p>

The identified reaction-curve values were approximately
$\tau_{ZN}=0.025$ s and $a=0.0588$, leading to the analytical starting
point shown in the next section.

The stored variable names `ki` and `kv` represented $T_i$ and $T_d$ in
this calculation, not the gains of a parallel-form PID. The distinction is
essential when entering values into a controller block.

In [ ]:
%% MATLAB implementation - reaction-curve parameters
% Use recorded vectors theta_l and t when available.
if exist('theta_l', 'var') == 1 && exist('t', 'var') == 1
    y = theta_l(:);
    tOut = t(:);
else
    [y, tOut] = step(speedPlant);
    warning(['Using an identified-model illustration because recorded ', ...
             'vectors are unavailable.']);
end
slope = gradient(y, tOut);
[maximumSlope, tangentIndex] = max(slope);

t0 = tOut(tangentIndex);
y0 = y(tangentIndex);
a = abs(maximumSlope*t0 - y0);
tauZN = abs(t0 - y0/maximumSlope);

KpZN = 1.2/a;
TiZN = 2*tauZN;
TdZN = 0.5*tauZN;

### Recorded result: iterative controller adjustment

The analytical starting point was approximately $K_p=20.4$,
$T_d=5\times10^{-4}$ s, and $T_i=0.002$ s under the notation used in the
project. It produced a fast response but drove the actuator into
saturation.

| Attempt 1 response | Attempt 1 voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/15-zn-attempt-1-step-response.png" width="430"> | <img src="../assets/results/position-control/16-zn-attempt-1-control-voltage.png" width="430"> |

Increasing the derivative contribution reduced the overshoot, but the
control voltage remained excessive.

| Attempt 2 response | Attempt 2 voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/17-zn-attempt-2-step-response.png" width="430"> | <img src="../assets/results/position-control/18-zn-attempt-2-control-voltage.png" width="430"> |

Reducing the proportional term to 10 removed the overshoot and restored
substantial voltage margin.

| Attempt 3 response | Attempt 3 voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/19-zn-attempt-3-step-response.jpg" width="430"> | <img src="../assets/results/position-control/20-zn-attempt-3-control-voltage.png" width="430"> |

The final recorded adjustment used approximately $K_p=12$ and a derivative
setting of 0.1. The reported rise time was 0.092 s without recorded
saturation.

| Final response | Final voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/21-zn-attempt-4-step-response.jpg" width="430"> | <img src="../assets/results/position-control/22-zn-attempt-4-control-voltage.png" width="430"> |

**Engineering interpretation.** The analytical rule supplied a useful starting point, but the first two settings exceeded the acceptable voltage range. Increasing damping and then reducing proportional action restored actuator margin before the final transient-speed adjustment.

### Recorded result: ramp and disturbance behavior

The recorded ramp tracking difference is approximately 0.047 rad. Under
manual disturbances, the tuned controller returns quickly to the command,
with visible voltage pulses corresponding to corrective motor action.

| Ramp response | Ramp voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/23-zn-ramp-response.jpg" width="430"> | <img src="../assets/results/position-control/24-zn-ramp-control-voltage.png" width="430"> |

| Disturbance response | Disturbance voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/25-zn-disturbance-response.jpg" width="430"> | <img src="../assets/results/position-control/26-zn-disturbance-control-voltage.png" width="430"> |

**Interpretation.** Reaction-curve tuning provided a useful starting point,
but the final safe controller depended on iterative adjustment. The stored
results demonstrate why analytical tuning values must still be checked
against actuator limits.

## 5. Relay auto-tuning

### Theory

A symmetric relay drives the closed loop into a sustained oscillation. If
$d$ is the relay amplitude, $a_y$ is the output oscillation amplitude, and
$T_c$ is the critical period, the describing-function approximation is

$$
K_c=\frac{4d}{\pi a_y},\qquad
\omega_c=\frac{2\pi}{T_c}.
$$

The complete relay tuning table used in the design is

| Controller | $K_p$ | $T_i$ | $T_d$ |
|---|---:|---:|---:|
| P | $0.5K_c$ | - | - |
| PI | $0.4K_c$ | $0.8T_c$ | - |
| PID | $0.6K_c$ | $0.5T_c$ | $0.125T_c$ |

The recorded oscillation produced $K_c\approx11.3861$ and
$T_c\approx0.3631$ s, giving the starting values $K_p\approx6.83$,
$T_i\approx0.18$ s, and $T_d\approx0.05$ s.

In [ ]:
%% MATLAB implementation - relay critical gain and period
load_system(fullfile(positionControlDir, 'RELAY_TUNING.slx'));
out = sim('RELAY_TUNING');
tRelay = out.tout(:);
yRelay = out.y(:);
uRelay = out.u(:);

startIndex = max(1, floor(numel(tRelay)/2));
relayAmplitude = (max(uRelay(startIndex:end)) - ...
                  min(uRelay(startIndex:end)))/2;
outputAmplitude = (max(yRelay(startIndex:end)) - ...
                   min(yRelay(startIndex:end)))/2;

[~, peakIndex] = findpeaks(yRelay(startIndex:end));
peakTime = tRelay(startIndex - 1 + peakIndex);
if numel(peakTime) < 2
    error('At least two steady relay-response peaks are required.');
end
periodCount = min(4, numel(peakTime) - 1);
Tc = mean(diff(peakTime(end-periodCount:end)));
Kc = 4*relayAmplitude/(pi*outputAmplitude);

KpRelay = 0.6*Kc;
TiRelay = 0.5*Tc;
TdRelay = 0.125*Tc;

### Recorded result and engineering interpretation

The initial controller had a 0.023 rad step-tracking difference (about 6%
for the test command), a rise time of 0.183 s, and a reported 5% settling
time of about 0.778 s.

| Initial response | Initial voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/27-relay-attempt-1-step-response.jpg" width="430"> | <img src="../assets/results/position-control/28-relay-attempt-1-control-voltage.png" width="430"> |

Increasing the proportional term to approximately 12 produced the fastest
recorded position response: 0.072 s rise time and 1.125% overshoot. The
voltage remained inside the operating limit but with less margin than LQR.

| Tuned response | Tuned voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/29-relay-attempt-2-step-response.jpg" width="430"> | <img src="../assets/results/position-control/30-relay-attempt-2-control-voltage.png" width="430"> |

The ramp tracking difference is approximately 0.044 rad. The disturbance
test shows recovery without sustained saturation.

| Ramp response | Ramp voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/31-relay-ramp-response.jpg" width="430"> | <img src="../assets/results/position-control/32-relay-ramp-control-voltage.png" width="430"> |

| Disturbance response | Disturbance voltage |
|:--:|:--:|
| <img src="../assets/results/position-control/33-relay-disturbance-response.png" width="430"> | <img src="../assets/results/position-control/34-relay-disturbance-control-voltage.png" width="430"> |

**Interpretation.** Relay auto-tuning offered the best recorded transient
after one additional proportional adjustment. Its cost was reduced voltage
margin, making saturation checks and hardware supervision especially
important.

## 6. Position-controller comparison

| Method | Recorded rise time | Recorded ramp difference | Main engineering trade-off |
|---|---:|---:|---|
| LQR | 0.174 s | 0.064 rad | Lowest control effort and straightforward model-based design |
| Tuned Ziegler-Nichols | 0.092 s | 0.047 rad | Fast response after several saturation-driven adjustments |
| Tuned relay | 0.072 s | 0.044 rad | Fastest response, but least voltage margin |

The comparison does not establish a universally superior controller. LQR
is attractive when model confidence and actuator margin dominate. The two
auto-tuning methods are attractive when a usable oscillation or step test
can be performed safely and rapid empirical adjustment is acceptable.

# Part II - SRV02-ROTFLEX direct digital control

## 7. Continuous flexible-joint model

### Theory

The state is

$$
x^T=\begin{bmatrix}\theta&\alpha&\dot\theta&\dot\alpha\end{bmatrix},
$$

and the linearized model is

$$
\dot{x}=Ax+Bu,\qquad y=Cx+Du.
$$

In sampled form, the state and output equations become

$$
x[k+1]=A_dx[k]+B_du[k],\qquad y[k]=C_dx[k]+D_du[k].
$$

The corresponding bilateral Z-transform definition is

$$
X(z)=\sum_{n=-\infty}^{\infty}x[n]z^{-n}.
$$

The matrices are assembled from the motor resistance $R_m$, torque and
back-EMF constants $k_t,k_m$, gear ratio $K_g$, efficiencies
$\eta_g,\eta_m$, equivalent inertia $J_{eq}$, arm inertia $J_{arm}$,
damping $B_{eq}$, and flexible-joint stiffness $K_s$.

In [ ]:
%% MATLAB implementation - continuous flexible-joint matrices
A = zeros(4,4);
A(1,3) = 1;
A(2,4) = 1;
A(3,2) = K_Stiff/Jeq;
A(3,3) = -(eta_g*Kg^2*eta_m*kt*km + Beq*Rm)/(Rm*Jeq);
A(4,2) = -K_Stiff*(Jeq + Jarm)/(Jeq*Jarm);
A(4,3) = (eta_g*Kg^2*eta_m*kt*km + Beq*Rm)/(Rm*Jeq);

B = zeros(4,1);
B(3) = eta_g*Kg*eta_m*kt/(Rm*Jeq);
B(4) = -B(3);

C = [1,0,0,0;
     0,1,0,0];
D = zeros(2,1);

### Computed result and engineering interpretation

For the stored high-gear, tachometer-equipped SRV02 and spring-2 ROTFLEX
configuration, the model reconstructs to approximately

$$
A=\begin{bmatrix}
0&0&1&0\\
0&0&0&1\\
0&483.131&-27.295&0\\
0&-1140.162&27.295&0
\end{bmatrix},\qquad
B=\begin{bmatrix}0\\0\\49.708\\-49.708\end{bmatrix},\qquad
C=\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix},\qquad
D=\begin{bmatrix}0\\0\end{bmatrix}.
$$

The fourth-row stiffness term is of order $10^3$, not $10^{-3}$. Its sign
and magnitude are confirmed by both the stored physical equations and the
discrete matrix below.

Key reconstructed parameters are:

| Parameter | Value | Unit |
|---|---:|---|
| $R_m$ | 2.6 | ohm |
| $k_t$ | 0.007683 | N m/A |
| $k_m$ | 0.007678 | V s/rad |
| $K_g$ | 70 | - |
| $J_{eq}$ | 0.002584 | kg m$^2$ |
| $J_{arm}$ | 0.001900 | kg m$^2$ |
| $K_s$ | 1.24850 | N m/rad (linearized rotational stiffness) |

The state coupling shows that spring deflection drives both angular accelerations with opposite signs. The corrected fourth-row stiffness magnitude is consistent with the stored physical equations and the exact discrete model.

## 8. Zero-order-hold discretization

### Theory

For a sample period $T_s$, zero-order hold assumes that the actuator command
is constant between samples. The exact discrete model is

$$
x[k+1]=A_dx[k]+B_du[k],\qquad
y[k]=C_dx[k]+D_du[k].
$$

The matrix exponential embedded in MATLAB `c2d(..., 'zoh')` preserves the
continuous plant evolution over each sampling interval.

In [ ]:
%% MATLAB implementation - exact ZOH discretization
Ts = 0.002;
continuousPlant = ss(A, B, C, D);
discretePlant = c2d(continuousPlant, Ts, 'zoh');
[Ad, Bd, Cd, Dd] = ssdata(discretePlant);

### Computed result and engineering interpretation

For $T_s=0.002$ s, executing the preceding cell produces

$$
A_d\approx\begin{bmatrix}
1&9.4856\!\times\!10^{-4}&1.9464\!\times\!10^{-3}&6.3533\!\times\!10^{-7}\\
0&0.997738&5.3590\!\times\!10^{-5}&1.99849\!\times\!10^{-3}\\
0&0.939641&0.946890&9.4856\!\times\!10^{-4}\\
0&-2.252710&0.053086&0.997738
\end{bmatrix},
$$

$$
B_d\approx\begin{bmatrix}
9.7615\!\times\!10^{-5}\\
-9.7593\!\times\!10^{-5}\\
0.096719\\
-0.096676
\end{bmatrix}.
$$

The position-state diagonal terms remain near one because the 2 ms interval is
short, while the off-diagonal terms preserve the flexible coupling. These are
computed model values rather than experimental measurements.

## 9. Direct digital pole placement

### Theory

A dominant continuous pole pair is selected from natural frequency
$\omega_n=20$ rad/s and damping ratio $\zeta=0.6$:

$$
s_{1,2}=-\zeta\omega_n\pm
j\omega_n\sqrt{1-\zeta^2}=-12\pm16j.
$$

Two faster real poles are placed at $-20$ and $-25$. Continuous poles are
mapped to the z-plane using

$$z_i=e^{s_iT_s}.$$

The same assignment can be obtained through canonical-form methods, direct
coefficient matching, or Ackermann's formula. The implementation uses MATLAB
`place` for numerical robustness.

In [ ]:
%% MATLAB implementation - discrete pole placement
zeta = 0.6;
wn = 20;
sigma = zeta*wn;
wd = wn*sqrt(1-zeta^2);

polesS = [-sigma + 1i*wd, -sigma - 1i*wd, -20, -25];
polesZ = exp(polesS*Ts);
Kpp = place(Ad, Bd, polesZ);
KI = [1.6, 1.6];

polesZ
Kpp

### Computed result and engineering interpretation

At $T_s=0.002$ s, executing the pole-placement cell gives

$$
z_{1,2}=0.975786\pm0.031236j,\quad
z_3=0.960789,\quad z_4=0.951229,
$$

and


$$K=\begin{bmatrix}5.876&-9.640&0.338&-0.461\end{bmatrix}.$$

A separately tuned integral path uses
$K_I=\begin{bmatrix}1.6&1.6\end{bmatrix}$. All requested poles lie inside
the unit circle, so the nominal discrete state-feedback dynamics are stable.
The integral path supplies the additional constant-reference tracking action.

## 10. Closed-loop architecture

<p align="center">
  <img src="../assets/diagrams/discrete-controller-model.jpg" width="980" alt="Digital controller Simulink architecture">
</p>

Signal flow, from left to right:

1. A quantized reference is converted from degrees to radians.
2. The reference is expanded to a four-state command vector.
3. The estimated state is subtracted from the state command.
4. The pole-placement gain $K$ produces the state-feedback action.
5. The measured position error is integrated using $T_s/(z-1)$.
6. The integral gain combines the motor-angle and deflection errors.
7. The two control contributions are summed and passed to the hardware
   plant subsystem.
8. The measured output $[\theta,\alpha]^T$ closes both the feedback and
   state-estimation paths.

The stored model implements the sign through a reference-minus-estimate
summing junction before the positive gain block. In an explicit equation,
the same convention is

$$u[k]=-K\hat{x}[k]+K_I\sum_{i=0}^{k}(r[i]-y[i])T_s.$$

## 11. Numerical state estimation

### Theory

Only $\theta$ and $\alpha$ are measured directly. The active path in
`Q_MDL_DISC.mdl` estimates velocity by backward difference:

$$
\dot\theta[k]\approx\frac{\theta[k]-\theta[k-1]}{T_s},\qquad
\dot\alpha[k]\approx\frac{\alpha[k]-\alpha[k-1]}{T_s}.
$$

This estimator is simple and introduces no model-dependent observer state,
but differentiation amplifies sensor quantization and noise. The alternate
`Q_CONTROL_Discret.mdl` keeps first-order high-pass derivative filters
active, trading some high-frequency attenuation for phase lag.

In [ ]:
%% MATLAB implementation - backward-difference state estimation
thetaDot = (theta(k) - theta(k-1))/Ts;
alphaDot = (alpha(k) - alpha(k-1))/Ts;
xEstimated = [theta(k); alpha(k); thetaDot; alphaDot];

integralError = integralError + (reference - measuredOutput)*Ts;
controlVoltage = -Kpp*xEstimated + KI*integralError;
controlVoltage = min(max(controlVoltage, -10), 10);

## 12. Recorded result and engineering interpretation: 2 ms behavior

**Recorded evidence — not regenerated because the raw measurements are unavailable.** The 2 ms controller follows the commanded motor angle with no meaningful
overshoot and a reported settling time of roughly 0.6 s. The flexible-
joint response remains centered near zero with transient deflection during
command reversals. The voltage capture shows the high-frequency content
expected from numerical differentiation and hardware measurement noise.

| Continuous motor angle | Discrete motor angle, 2 ms |
|:--:|:--:|
| <img src="../assets/results/flexible-joint/35-continuous-motor-angle-response.jpg" width="430"> | <img src="../assets/results/flexible-joint/36-discrete-motor-angle-ts-0p002.jpg" width="430"> |

| Continuous deflection | Discrete deflection, 2 ms |
|:--:|:--:|
| <img src="../assets/results/flexible-joint/37-continuous-flexible-angle-response.jpg" width="430"> | <img src="../assets/results/flexible-joint/38-discrete-flexible-angle-ts-0p002.jpg" width="430"> |

<p align="center">
  <img src="../assets/results/flexible-joint/39-discrete-control-voltage-ts-0p002.jpg" width="720" alt="Recorded discrete control voltage at a 2 ms sample period">
</p>

**Interpretation.** At 500 Hz, the digital implementation samples much
faster than the dominant closed-loop dynamics. The result is close to the
continuous design, although the numerical derivative is visibly sensitive
to measurement noise.

## 13. Sampling-time sensitivity

### Theory

The nominal controller is designed at 2 ms. Executing the same fixed gain at
slower rates changes the discrete plant seen by the controller, adds effective
sensing and actuation delay, and worsens backward-difference velocity
estimation. The following offline implementation explores those mechanisms; it
does not recreate the recorded hardware captures.

In [ ]:
%% MATLAB implementation - fixed-gain sampling-time comparison
% Supplemental simulation; recorded figures are presented below.
run(fullfile(repositoryRoot, 'SRV02-Observer-Control', ...
    'sampling_time_comparison.m'));

### Recorded result and engineering interpretation

**Recorded evidence — not regenerated because the raw measurements are unavailable.** At 50 ms, the recorded motor and deflection responses remain controlled,
but the transitions are more oscillatory and the effective delay is much
larger relative to the plant dynamics.

| Motor angle, 50 ms | Flexible deflection, 50 ms |
|:--:|:--:|
| <img src="../assets/results/flexible-joint/40-discrete-motor-angle-ts-0p05.jpg" width="430"> | <img src="../assets/results/flexible-joint/41-discrete-flexible-angle-ts-0p05.jpg" width="430"> |

At 100 ms, the recorded experiment loses control:

<p align="center">
  <img src="../assets/results/flexible-joint/42-discrete-response-ts-0p1.jpg" width="720" alt="Recorded unstable response at a 100 ms sample period">
</p>

| Sample period | Sampling rate | Recorded behavior |
|---:|---:|---|
| 0.002 s | 500 Hz | Stable and close to the continuous response |
| 0.05 s | 20 Hz | Stable, slower, and more oscillatory |
| 0.1 s | 10 Hz | Loss of control |

**Supplementary engineering explanation.** Increasing $T_s$ adds effective
sensing and actuation delay, worsens the numerical derivative, and reduces
the separation between the sampling rate and the flexible mode. A practical
sample period should be selected from the fastest relevant closed-loop and
structural dynamics, then validated with computation time, quantization,
saturation, and hardware safety included. The documented engineering rule
of thumb was to sample at least ten times faster than the characteristic
system time scale; this is a starting criterion, not a substitute for the
closed-loop and hardware checks above.

## 14. Supplemental observer and Kalman comparison

### Theory

`observer_estimator_comparison.m` provides an offline extension that is not
part of the recorded experiment. It compares:

- Backward numerical differentiation.
- A discrete Luenberger observer

  $$\hat{x}[k+1]=A_d\hat{x}[k]+B_du[k]
  +L\left(y[k]-C_d\hat{x}[k]\right).$$

- A discrete Kalman filter with explicitly declared process and measurement
  covariance matrices.

A documented sensitivity sweep proposed synthetic angular-noise levels of
$0$, $0.001$, $0.005$, and $0.01$ rad. No measured result set accompanied
that sweep, so it remains an analysis recipe rather than experimental
evidence. The repository script instead uses a declared, reproducible noise
level and labels its figures accordingly. Its purpose is to explore
estimator noise sensitivity, not to create additional experimental evidence.

In [ ]:
%% MATLAB implementation - Luenberger observer gain
observerPolesS = [-55, -60, -65, -70];
observerPolesZ = exp(observerPolesS*Ts);
L = place(Ad', Cd', observerPolesZ)';

% Discrete Luenberger prediction with measurement innovation.
innovation = measuredOutput - Cd*xEstimated;
xEstimatedNext = Ad*xEstimated + Bd*controlVoltage + L*innovation;

### Computed result and engineering interpretation

Running `observer_estimator_comparison.m` generates motor-angle, velocity-
estimate, estimation-error, and control-effort plots together with velocity
RMSE and sustained 5% settling-time metrics. Because MATLAB is unavailable in
this documentation environment, no numerical output is embedded here.

These outputs are explicitly supplemental simulations with reproducible
synthetic noise, not recorded experimental evidence. Their engineering purpose
is to expose the noise amplification of backward differentiation and compare it
with model-based state estimation under identical conditions.

## 15. Theory-to-code map

| Engineering concept | Main implementation |
|---|---|
| SRV02 speed model | `SRV02-Position-Control/Q2_ZN.m` |
| LQR cost and gain | `SRV02-Position-Control/LQR_Controller.m` |
| Relay oscillation and critical gain | `Q2_RELAY.m`, `RELAY_TUNING.slx` |
| SRV02 and ROTFLEX physical constants | `Modules/config_srv02.m`, `Modules/config_rotflex.m` |
| Continuous $A,B,C,D$ equations | `Modules/SRV02_ROTFLEX_ABCD_eqns.m` |
| Nominal pole mapping and $K$ | `SRV02-Observer-Control/model3.m` |
| Numerical-derivative controller | `Q_MDL_DISC.mdl` |
| Filtered-derivative variant | `Q_CONTROL_Discret.mdl` |
| Fixed-gain sampling illustration | `sampling_time_comparison.m` |
| Estimator extension | `observer_estimator_comparison.m` |

The two offline comparison scripts also preserve the implementation details
for the three-period sampling loop, numerical velocity reconstruction,
z-plane pole plots with the unit circle, synthetic-noise evaluation,
steady-state error, sustained 5% settling time, and the custom discrete
state-update simulation.

## 16. Reproduction workflow

### Position-control scripts

```matlab
cd('SRV02-Position-Control')
run('LQR_Controller.m')
run('Q2_ZN.m')       % Recorded theta_l and t vectors are preferred.
run('Q2_RELAY.m')    % Requires Simulink and findpeaks.
```

### Flexible-joint model

```matlab
cd('SRV02-Observer-Control')
run('model3.m')
open_system('Q_MDL_DISC')
```

`model3.m` adds the `Modules` folder to the MATLAB path, initializes the
stored high-gear ROTFLEX-E setup, computes $A,B,C,D$, discretizes at 2 ms,
and places the nominal poles.

### Hardware checklist

1. Install a MATLAB release compatible with the selected QUARC release.
2. Confirm the Q2-USB board and channel assignments.
3. Confirm potentiometer/encoder selection and direction.
4. Confirm amplifier type and gain.
5. Keep analog output disabled while validating the model.
6. Check the $\pm10$ V command limit, mechanical travel, and emergency stop.
7. Enable output only under direct supervision.

## 17. Limitations

- Raw time-series experiment data are not present. Recorded plots and
  reported metrics can be preserved and interpreted, but not all can be
  independently recomputed.
- The hardware models require proprietary Quanser libraries and compatible
  hardware. Without them, library-link warnings are expected.
- The stored model variants do not use identical derivative filters or
  integral gains. The numerical-derivative `Q_MDL_DISC.mdl` path is treated
  as the nominal documented design.
- The Ziegler-Nichols reaction-curve script needs recorded response vectors
  for meaningful experimental tuning. A pure first-order fallback is only
  an illustration and may not contain a resolvable process delay.
- The recorded comparison mixes simulation/model runs and hardware captures
  without raw metadata that would allow every figure to be classified more
  precisely.
- The supplemental simulations require local MATLAB execution and are not
  presented as measured results.

## 18. Engineering conclusions

1. Model-based LQR provided a strong first design with the best recorded
   voltage margin and no repeated tuning cycle.
2. Reaction-curve and relay methods produced faster final responses, but
   safe performance depended on checking saturation and adjusting the
   analytical starting values.
3. The flexible-joint state-space model, ZOH conversion, pole mapping, and
   gain matrix are internally consistent and reproducible from the stored
   Quanser parameters.
4. Numerical differentiation is adequate at the nominal 2 ms sample period,
   but its noise amplification is visible and motivates filtered derivatives
   or a model-based observer when measurement quality is poor.
5. Sampling time is a design variable, not an implementation detail. The
   recorded transition from stable 2 ms and 50 ms control to loss of control
   at 100 ms demonstrates the need to include computation, delay, flexible
   dynamics, and saturation in digital-controller validation.